# Train bộ 18 run trên Colab, xen kẽ với máy 5090 (26/11)

1. Runtime **GPU** (L4). Trong 🔑 Secrets bật `GH_TOKEN` cho notebook này.
2. Chạy Setup (giải nén ảnh khoảng 10 phút), rồi cell I (nhận zip mới của máy 5090 trong `MyDrive/Sym/upload/`; không có thì bỏ qua), rồi cell G2 (train + refit, kèm % và giờ xong dự kiến).
3. Khi muốn chuyển sang máy 5090: dừng cell G2, chạy cell M, tải các file trong `MyDrive/Sym/handoff/` sang `D:\LVTS_TriLe\nhan` trên máy 5090.

Mỗi lúc chỉ một nơi train một run: nơi gửi đánh dấu run dở là đã bàn giao và không train tiếp nó. Không đụng test.

In [ ]:
import os, sys, json, time, base64, shutil, subprocess
from pathlib import Path
from google.colab import drive, userdata

if not os.path.isdir('/content/drive/MyDrive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/Letri6102/Thesis_Sym.git'
BRANCH   = 'main'                      # nhánh stage1-ema-valchecks đã merge vào main (21/11)
REPO     = Path('/content/Thesis_Sym')
CODE     = REPO / 'Contrastive_V1'
SYM      = Path('/content/drive/MyDrive/Sym')
SPLITS   = SYM / 'splits_v2'           # manifest mới; splits/ cũ chỉ cho runs_retry1
RUNS     = SYM / 'runs_aug1'           # run mới với strong_train_aug
LOGS     = SYM / 'logs'
IMAGES   = True                        # False: không giải nén ảnh (phiên CPU chỉ chạy Tầng 2, cell E)

tok  = userdata.get('GH_TOKEN')
auth = 'http.extraHeader=Authorization: Basic ' + base64.b64encode(f'x-access-token:{tok}'.encode()).decode()
def git(*args, cwd=None):
    r = subprocess.run(['git', '-c', auth, *args], cwd=cwd, capture_output=True, text=True,
                       env={**os.environ, 'GIT_TERMINAL_PROMPT': '0'})
    if r.returncode:
        raise RuntimeError(f'git {args[0]} lỗi (exit {r.returncode}): {r.stderr.strip()}')
try:
    if (REPO / '.git').exists():
        # refspec đầy đủ: bản clone 1 nhánh vẫn đổi được sang nhánh khác (fetch tên trơn chỉ cập nhật FETCH_HEAD)
        git('fetch', '-q', '--depth', '1', 'origin', f'+refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}', cwd=REPO)
        git('reset', '-q', '--hard', f'origin/{BRANCH}', cwd=REPO)
    else:
        git('clone', '-q', '--depth', '1', '-b', BRANCH, REPO_URL, str(REPO))
finally:
    del tok, auth
print('commit:', subprocess.run(['git', 'log', '-1', '--format=%h %s'], cwd=REPO,
                                capture_output=True, text=True).stdout.strip())

SHARDS = SYM / 'DISFA_img'                     # 27 tar theo subject, do cell A tạo
if IMAGES and (SHARDS / 'DONE.json').exists():
    DATA = Path('/content/DISFA')
    if not (DATA / '.img_ok').exists():        # chưa giải nén, hoặc lần trước giải nén dở
        t0, meta = time.time(), json.loads((SHARDS / 'DONE.json').read_text())
        DATA.mkdir(parents=True, exist_ok=True)
        print(f'Giải nén {len(meta["subjects"])} tar ({meta["bytes"] / 1e9:.1f} GB) về /content:', flush=True)
        for i, s in enumerate(meta['subjects'], 1):
            subprocess.run(['tar', '-xf', str(SHARDS / f'{s}.tar'), '-C', str(DATA)], check=True, timeout=1800)
            print(f'  {i:>2}/{len(meta["subjects"])} {s}  {(time.time() - t0) / 60:.1f} phút', flush=True)
        n = sum(len(f) for _, _, f in os.walk(DATA / 'img'))
        assert n == meta['n_images'], f'Giải nén được {n} ảnh, cần {meta["n_images"]}'
        shutil.copytree(SYM / 'DISFA' / 'list', DATA / 'list', dirs_exist_ok=True)
        (DATA / '.img_ok').touch()
        print(f'Đã giải nén {n} ảnh về /content trong {time.time() - t0:.0f}s')
else:
    DATA = SYM / 'DISFA'                 # đọc thẳng Drive: chậm và hay lỗi FileNotFoundError
print('DATA =', DATA)

os.chdir(CODE)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
(CODE / 'checkpoints').mkdir(exist_ok=True)
w_src = SYM / 'Contrastive_V1' / 'checkpoints' / 'resnet50-19c8e357.pth'
w_dst = CODE / 'checkpoints' / 'resnet50-19c8e357.pth'
if w_src.exists() and not w_dst.exists():
    shutil.copy(w_src, w_dst)
import torch, torchvision
print('torch', torch.__version__, '| torchvision', torchvision.__version__, '| CPU', os.cpu_count(),
      '| GPU', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ')

import re
def run_logged(cmd, name):
    """Chạy lệnh con, ghi toàn bộ output vào MyDrive/Sym/logs/ và in các dòng đáng xem.
    (Colab không hiện stdout của tiến trình con nếu chỉ dùng subprocess.run.)"""
    LOGS.mkdir(exist_ok=True)
    log = LOGS / f'{time.strftime("%Y%m%d_%H%M%S")}_{name}.log'
    print(f'>> {name} | log: {log.name}', flush=True)
    with open(log, 'w') as f, subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE,
                                               stderr=subprocess.STDOUT, text=True, bufsize=1) as p:
        for line in p.stdout:
            f.write(line); f.flush()
            if line.startswith('{"epoch"'):
                r = json.loads(line)
                print(f"   ep{r['epoch']:>2}  wa={r['train']['wa']:.4f}  valF1={r['val']['macro_f1']:.3f}"
                      f"  {r['seconds'] / 60:.0f} phút", flush=True)
            elif line.startswith('epoch ') and ' step ' in line:
                if re.search(r'step (1000|2000)/', line):
                    print('   ' + line.rstrip(), flush=True)
            elif 'FutureWarning' not in line and 'warnings.warn' not in line:
                print('   ' + line.rstrip(), flush=True)
    if p.returncode:
        raise RuntimeError(f'{name} dừng với mã {p.returncode}, xem log {log}')

## Cell I — nhận run (xong và dở) từ máy 5090

In [ ]:
# ==== Cell I: nhận run từ máy 5090 (MyDrive/Sym/upload/*.zip): run xong và run dở — kiểm khoá, dòng lịch sử, cặp ====
sys.path.insert(0, str(REPO / 'pc'))
from stage2_decoder.contrast_ci import LOCK
import train_18_run as PC
PC.receive(SYM / 'upload', out=SYM, splits=SPLITS, data=DATA, lock=LOCK, work=Path('/content/pc_import'))
for kind, root in PC.KIND_DIR.items():
    n = sum(PC._done(kind, SYM / root / v / f'fold{k}' / f'seed{s}', LOCK['common']['epochs']) for v, k, s in PC.jobs())
    print(f'{root}: {n}/18 run xong', flush=True)
print('Đang chờ Drive đồng bộ...', flush=True)
drive.flush_and_unmount()

## Cell G2 — train trên Colab

In [ ]:
# ==== Cell G2: train bộ 18 run trên Colab bằng đúng runner của máy 5090 (26/11). Drive là bản chính. KHÔNG đụng test ====
if not os.path.isdir('/content/drive/MyDrive'):            # cell I/M tháo Drive ở cuối: mount lại
    drive.mount('/content/drive')
assert str(DATA).startswith('/content/'), 'Ảnh chưa nằm trong /content: chạy Setup với IMAGES = True'
shutil.copytree(SPLITS, DATA / 'splits_v2', dirs_exist_ok=True)       # runner đọc manifest ở <DATA>/splits_v2 (có kiểm hash khoá)
SEEDS, FOLDS = [0, 1, 2], [1, 2, 3]
REFIT = True                     # xong (a) thì refit luôn những run (a) đã xong
OLD_SESSION_DEAD = False         # chỉ đặt True khi chắc chắn phiên Colab trước đã tắt hẳn
RECLAIM = False                  # chỉ đặt True khi lấy lại run đã bàn giao mà máy 5090 CHƯA train tiếp
args = ['--data', DATA, '--out', SYM, '--seeds', *SEEDS, '--folds', *FOLDS, '--no-pack']
args += ['--old-session-dead'] * OLD_SESSION_DEAD + ['--reclaim'] * RECLAIM
runner = [sys.executable, '-u', REPO / 'pc' / 'train_18_run.py']
run_logged(runner + ['train', *args], 'colab_train')
if REFIT:
    run_logged(runner + ['refit', *args], 'colab_refit')
run_logged(runner + ['progress', '--out', SYM], 'colab_progress')

## Cell M — bàn giao cho máy 5090 (dừng G2 trước)

In [ ]:
# ==== Cell M: bàn giao cho máy 5090 — zip run dở (partial_...) + bản tóm tắt run đã xong (stubs_...) vào MyDrive/Sym/handoff ====
if not os.path.isdir('/content/drive/MyDrive'):            # cell I/M tháo Drive ở cuối: mount lại
    drive.mount('/content/drive')
# Dừng cell G2 trước. Run dở được đánh dấu HANDED_OFF trên Drive: Colab không train tiếp cho tới khi nhận bản mới về (cell I).
STOPPED_HERE = False             # True: vừa tự dừng cell G2 trong phiên này, không cần chờ 15 phút
sys.path.insert(0, str(REPO / 'pc'))
from stage2_decoder.contrast_ci import LOCK
import train_18_run as PC
OUT_DIR = SYM / 'handoff'
for old in OUT_DIR.glob('*.zip'):                         # bản bàn giao cũ chưa dùng: cất đi, tránh nhận nhầm
    (OUT_DIR / 'cu').mkdir(parents=True, exist_ok=True)
    shutil.move(str(old), str(OUT_DIR / 'cu' / old.name))
PC.send(out=SYM, dest=OUT_DIR, complete='stub', lock=LOCK, work=Path('/content/handoff_work'),
        old_session_dead=STOPPED_HERE, to='máy 5090')
print('Tải các file trên trong MyDrive/Sym/handoff về máy cá nhân, chép vào D:\\LVTS_TriLe\\nhan trên máy 5090,'
      ' rồi chạy train_data_co_san.bat (tự nhận trước khi train).', flush=True)
drive.flush_and_unmount()